In [12]:
import torch
from transformers import RobertaTokenizer, RobertaConfig, RobertaModel, AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM, Trainer, TrainingArguments
from tqdm import tqdm
import os
from datasets import Dataset
import random

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
DATASET_ROOT = "../../CrossVul"
ALLOWED_CWE_IDS = {"CWE-89"} # "CWE-79", "CWE-89", "CWE-787"
LANGUAGES = ['c', 'cpp', 'cs', 'html', 'java', 'py', 'php']

In [13]:
vulBERTa = "claudios/VulBERTa-MLP-ReVeal"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(vulBERTa, trust_remote_code=True)
print(device)

cuda


# Model Finetuning:

In [ ]:
def collect_files_for_cwe(cwe_id):
    samples = []
    for lang in LANGUAGES:
        lang_dir = os.path.join(DATASET_ROOT, cwe_id, lang)
        if not os.path.isdir(lang_dir):
            continue
        for filename in os.listdir(lang_dir):
            filepath = os.path.join(lang_dir, filename)
            if filename.endswith('.DS_Store'):
                continue
            label = 1 if "bad" in filename.lower() else 0
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                code = f.read()
            samples.append({
                "filename": filename,
                "code": code,
                "label": label
            })
    return samples

def tokenize_example(example):
    prompt = "Does this code have a vulnerability relating to CWE-22 Path Traversal?"
    encoded = tokenizer(
        prompt,
        example["code"],
        padding="max_length",
        max_length=512,
        truncation=True
    )
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "label": example["label"]
    }


for cwe_id in ALLOWED_CWE_IDS:
    print(f"\n--- Training for {cwe_id} ---")

    samples = collect_files_for_cwe(cwe_id)
    random.shuffle(samples)

    raw_dataset = Dataset.from_list(samples)
    tokenized_dataset = raw_dataset.map(tokenize_example)

    train_test = tokenized_dataset.train_test_split(test_size=0.2)
    train_dataset = train_test["train"]
    eval_dataset = train_test["test"]

    model = AutoModelForSequenceClassification.from_pretrained(vulBERTa, num_labels=2).to(device)

    training_args = TrainingArguments(
        output_dir=f"./models/vulberta_{cwe_id}",
        evaluation_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=4,
        weight_decay=0.01,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    trainer.train()
    trainer.save_model(f"./models/vulberta_{cwe_id}/final")

# Model Evaluation:

In [7]:
prompt_tokens = tokenizer.tokenize("Does this code have a vulnerability?")
# int main() { int c = 2; int arr[] = {1}; printf(\"%d\", arr[c]);}
file = open('testfile.c', 'r')
code = file.read()
print(code)
code_tokens = tokenizer.tokenize(code)
inputs = tokenizer.encode_plus("".join(prompt_tokens), "".join(code_tokens), add_special_tokens=True, padding=True, truncation=True, return_tensors="pt")
classification_model = AutoModelForSequenceClassification.from_pretrained(vulBERTa)
classification_model.to(device)

for input_data in tqdm([inputs], desc="Processing batch", unit="example"):
    input_ids = inputs['input_ids'].to(device)
    attention_mask = inputs['attention_mask'].to(device)
    classification_model.eval()
    with torch.no_grad():
        outputs = classification_model(input_ids=input_ids, attention_mask=attention_mask)

logits = outputs.logits
predicted_class = torch.argmax(logits, dim=1).item()
if predicted_class == 1:
    print("The code has a vulnerability.")
else:
    print("The code does not have a vulnerability.")

FileNotFoundError: [Errno 2] No such file or directory: 'testfile.c'